# 🎗️ Breast Cancer Prediction
**Author:** Bhardwaja  
**Dataset:** UCI Wisconsin Breast Cancer Dataset (569 samples, 30 features)  
**Goal:** Classify tumours as Malignant (M) or Benign (B) using cell nucleus measurements.

---
### Pipeline Overview
1. Import Libraries
2. Load & Explore Data (EDA)
3. Data Cleaning & Preprocessing
4. Train/Test Split (No Data Leakage)
5. Model Training & Comparison
6. Evaluation & Visualisation
7. Save Best Model

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve
)
import joblib

sns.set(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
print('All libraries imported successfully!')

## 2. Load & Explore the Data

In [ ]:
df = pd.read_csv('breast_cancer_data.csv')
print('Dataset Shape:', df.shape)
df.head(10)

In [ ]:
print('=== Dataset Info ===')
df.info()

In [ ]:
print('=== Statistical Summary ===')
df.describe()

In [ ]:
# Class distribution
counts = df['diagnosis'].value_counts()
print('Malignant (M):', counts['M'], f'({counts["M"]/len(df)*100:.1f}%)')
print('Benign    (B):', counts['B'], f'({counts["B"]/len(df)*100:.1f}%)')

plt.figure(figsize=(6, 4))
counts.plot(kind='bar', color=['#e74c3c', '#2ecc71'], edgecolor='black')
plt.title('Class Distribution: Malignant vs Benign', fontsize=14)
plt.xlabel('Diagnosis')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Check for missing values
print('Missing values per column:')
print(df.isnull().sum())
print(f'\nTotal missing values: {df.isnull().sum().sum()}')

In [ ]:
# Feature distributions: mean features by diagnosis
mean_features = [c for c in df.columns if '_mean' in c]

fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()

for i, feat in enumerate(mean_features):
    m_vals = df[df['diagnosis']=='M'][feat]
    b_vals = df[df['diagnosis']=='B'][feat]
    axes[i].hist(m_vals, alpha=0.6, label='Malignant', color='#e74c3c', bins=20)
    axes[i].hist(b_vals, alpha=0.6, label='Benign', color='#2ecc71', bins=20)
    axes[i].set_title(feat.replace('_mean','').replace('_',' ').title(), fontsize=10)
    axes[i].legend(fontsize=8)

plt.suptitle('Mean Feature Distributions by Diagnosis', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap — mean features only (30 features is too many to show all)
df_temp = df[mean_features + ['diagnosis']].copy()
df_temp['diagnosis_enc'] = (df_temp['diagnosis'] == 'M').astype(int)

plt.figure(figsize=(14, 10))
corr = df_temp.drop('diagnosis', axis=1).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', linewidths=0.3, annot_kws={'size': 7}
)
plt.title('Correlation Heatmap — Mean Features', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot: top separating features
top_feats = ['radius_mean', 'perimeter_mean', 'area_mean', 'concavity_mean', 'concave_points_mean']

fig, axes = plt.subplots(1, 5, figsize=(20, 5))
for ax, feat in zip(axes, top_feats):
    sns.boxplot(
        x='diagnosis', y=feat, data=df, ax=ax,
        palette={'M': '#e74c3c', 'B': '#2ecc71'}
    )
    ax.set_title(feat.replace('_mean','').replace('_',' ').title(), fontsize=11)
plt.suptitle('Key Features: Malignant vs Benign', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 3. Data Cleaning & Preprocessing

In [ ]:
# Drop 'id' — not a feature
df.drop('id', axis=1, inplace=True)

# Encode target: Malignant=1, Benign=0
df['diagnosis'] = df['diagnosis'].map({'M': 1, 'B': 0})

X = df.drop('diagnosis', axis=1)
y = df['diagnosis']

print('Features shape:', X.shape)
print('Target distribution:')
print(y.value_counts())
print(f'Malignant: {y.sum()} ({y.mean()*100:.1f}%)  |  Benign: {(y==0).sum()} ({(y==0).mean()*100:.1f}%)')

## 4. Train/Test Split & Scaling (No Data Leakage)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set:     {X_test.shape[0]} samples')

# Fit scaler ONLY on training data — transform test separately
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('\nScaling complete — no data leakage.')

## 5. Model Training & Comparison

Training 4 models:
- **Logistic Regression** — strong interpretable baseline for medical data
- **Random Forest** — handles feature interactions well
- **Support Vector Machine (SVM)** — excellent for high-dimensional data like this
- **Gradient Boosting** — powerful ensemble method

In [ ]:
# --- Model 1: Logistic Regression ---
lr = LogisticRegression(max_iter=10000, random_state=42)
lr.fit(X_train_scaled, y_train)
lr_pred = lr.predict(X_test_scaled)
lr_acc  = accuracy_score(y_test, lr_pred)

print(f'Logistic Regression Accuracy: {lr_acc:.4f}')
print(classification_report(y_test, lr_pred, target_names=['Benign','Malignant']))

In [ ]:
# --- Model 2: Random Forest ---
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_acc  = accuracy_score(y_test, rf_pred)

print(f'Random Forest Accuracy: {rf_acc:.4f}')
print(classification_report(y_test, rf_pred, target_names=['Benign','Malignant']))

In [ ]:
# --- Model 3: SVM ---
svm = SVC(kernel='rbf', probability=True, random_state=42)
svm.fit(X_train_scaled, y_train)
svm_pred = svm.predict(X_test_scaled)
svm_acc  = accuracy_score(y_test, svm_pred)

print(f'SVM Accuracy: {svm_acc:.4f}')
print(classification_report(y_test, svm_pred, target_names=['Benign','Malignant']))

In [ ]:
# --- Model 4: Gradient Boosting ---
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb.fit(X_train, y_train)
gb_pred = gb.predict(X_test)
gb_acc  = accuracy_score(y_test, gb_pred)

print(f'Gradient Boosting Accuracy: {gb_acc:.4f}')
print(classification_report(y_test, gb_pred, target_names=['Benign','Malignant']))

## 6. Evaluation & Visualisation

In [ ]:
# Accuracy comparison
model_names = ['Logistic Reg', 'Random Forest', 'SVM', 'Gradient Boost']
accuracies  = [lr_acc, rf_acc, svm_acc, gb_acc]

plt.figure(figsize=(9, 5))
bars = plt.bar(model_names, accuracies,
               color=['#9b59b6','#e67e22','#3498db','#e74c3c'],
               edgecolor='black')
for bar, acc in zip(bars, accuracies):
    plt.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() - 0.03,
        f'{acc:.2%}', ha='center', va='bottom',
        fontsize=12, fontweight='bold', color='white'
    )
plt.ylim(0.85, 1.01)
plt.title('Model Accuracy Comparison — Breast Cancer Prediction', fontsize=14)
plt.ylabel('Accuracy')
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
model_info = [
    ('Logistic Reg', lr_pred),
    ('Random Forest', rf_pred),
    ('SVM', svm_pred),
    ('Gradient Boost', gb_pred)
]
for ax, (name, pred) in zip(axes, model_info):
    cm = confusion_matrix(y_test, pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Benign','Malignant'])
    disp.plot(ax=ax, cmap='Reds', colorbar=False)
    ax.set_title(f'{name}\nAcc: {accuracy_score(y_test,pred):.2%}', fontsize=11)
plt.suptitle('Confusion Matrices', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curves
plt.figure(figsize=(9, 6))
roc_models = [
    ('Logistic Reg', lr, X_test_scaled),
    ('Random Forest', rf, X_test),
    ('SVM', svm, X_test_scaled),
    ('Gradient Boost', gb, X_test),
]
for name, model, Xte in roc_models:
    proba = model.predict_proba(Xte)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')

plt.plot([0,1],[0,1],'k--', label='Random Chance')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curves — Breast Cancer Prediction', fontsize=14)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance — Random Forest (top 15)
feat_imp = pd.Series(
    rf.feature_importances_, index=X.columns
).sort_values(ascending=False).head(15)

plt.figure(figsize=(11, 6))
sns.barplot(x=feat_imp.values, y=feat_imp.index, palette='magma')
plt.title('Top 15 Feature Importances — Random Forest', fontsize=14)
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

print('Top 5 most important features:')
print(feat_imp.head())

In [ ]:
# Cross-validation on best model
best_model_cv = svm  # likely SVM or LR for this dataset
cv_scores = cross_val_score(best_model_cv, X_train_scaled, y_train, cv=5, scoring='accuracy')
print(f'SVM 5-Fold CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')
print(f'Individual fold scores: {np.round(cv_scores, 4)}')

## 7. Save Best Model

In [ ]:
# Save the best model and scaler
# Update 'best_model' below based on which model won above
best_model = svm  # change if another model performs better

joblib.dump(best_model, 'breast_cancer_model.pkl')
joblib.dump(scaler, 'breast_cancer_scaler.pkl')

print('Saved: breast_cancer_model.pkl')
print('Saved: breast_cancer_scaler.pkl')
print('\nReady for the multi-disease Streamlit app!')

## 8. Summary

| Model | Test Accuracy |
|---|---|
| Logistic Regression | see above |
| Random Forest | see above |
| **SVM (RBF kernel)** | **likely best** |
| Gradient Boosting | see above |

**Key Findings:**
- `concave_points_worst`, `perimeter_worst`, and `radius_worst` are the strongest predictors — size and shape of the nucleus at its worst (largest) cross-section is most discriminative.
- SVM with RBF kernel typically achieves the best AUC on this dataset due to the high dimensionality (30 features).
- No missing values in this dataset — no imputation needed.

**Possible improvements:**
- PCA for dimensionality reduction (30 features → 10 principal components)
- Hyperparameter tuning: SVM `C` and `gamma` via GridSearchCV
- Threshold tuning: in cancer detection, minimising False Negatives (missed cancers) is more important than raw accuracy